# SAM 2.1 count test on the WGISD test photos

Runs the automatic SAM 2.1 pipeline (**detector → SAM 2.1**) on the 27 WGISD test photos, counts the grape clusters it finds, and compares the counts with the hand counts below. The detector's own count is shown next to it, so the table says what SAM 2.1 adds. It is the SAM 2.1 counterpart of `yolo26_pipeline/count_test.py`, on the same photos and the same hand counts.

**How to run:** the same way as `demo.ipynb` (see `INSTALL.md`): `conda activate ./envs`, then `jupyter notebook` **from this folder** (`sam2_pipeline/`), open this notebook and *Run ▸ Run All Cells*. The first run installs `ultralytics` into the environment and downloads the SAM 2.1 weights (about 150 MB). A laptop CPU needs a few minutes for the 27 photos.

Everything it writes goes to `data/count_test/` (git ignores it): the table in `results/count_test.csv`, the metrics in `results/count_metrics.csv`, one overlay image per photo in `results/overlays/`.

In [ ]:
# Step 1: load the pipeline code (fruit_sam2.py, in this folder) and point it at a scratch project folder.
import os, sys, shutil, subprocess
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# Opened in Google Colab straight from GitHub? Then fetch the repository first and work inside its sam2_pipeline folder.
BRANCH = "main"            # the branch of the team repository to fetch
if not Path("fruit_sam2.py").exists() and "google.colab" in sys.modules:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    "https://github.com/leonorasylejmani7250/Capstone-MSU_Fruit.git", "/content/Capstone-MSU_Fruit"], check=True)
    os.chdir("/content/Capstone-MSU_Fruit/sam2_pipeline")
if not Path("fruit_sam2.py").exists() and Path("sam2_pipeline/fruit_sam2.py").exists():
    os.chdir("sam2_pipeline")      # Jupyter was started from the repository root instead of sam2_pipeline/
assert Path("fruit_sam2.py").exists(), "Start Jupyter from the sam2_pipeline folder (see INSTALL.md)."

sys.path.insert(0, str(Path.cwd()))
import fruit_sam2 as F          # first run: fetches Meta's SAM 2 code and installs the packages it needs

F.SAM2_SIZE = "tiny"            # tiny | small | base_plus | large (base_plus is the pipeline's default; needs a GPU to be quick)
if F.DEVICE == "cpu":
    F.SAM2_IMAGE_SIZE = 512     # faster on a CPU, a little less accurate; GPUs use SAM 2's full 1024
F.use_project("../data/count_test")                         # the repository's shared data folder; this test keeps to its own subfolder

IMAGES = Path("../data/wgisd_yolo/images/test").resolve()  # the 27 WGISD test photos (never trained on)
assert IMAGES.is_dir(), f"missing {IMAGES}: the WGISD photos should be in data/wgisd_yolo (see data/README.md)"
print("running on", F.GPU_NAME or "CPU", "· photos:", IMAGES, "· results:", F.ROOT)

## The detector
SAM 2.1 does not find grapes by itself: it segments and follows what it is shown. In the automatic pipeline a detector finds the clusters and SAM 2.1 turns each box into a mask (on video it then tracks it from frame to frame). The default below is the team's **YOLO26** trained on WGISD, `yolo26_pipeline/model/best.pt`, which is in the repository and needs no download; it uses the same settings as `yolo26_pipeline/count_test.py` (image size 512, confidence 0.25), so the "YOLO26 boxes" column should match that script's results. The commented line is the zero-shot alternative, YOLOE with a text prompt.

In [ ]:
# Step 2: the detector that finds clusters, and the detector → SAM 2.1 pipeline built on it.
DETECTOR = F.UltralyticsModel("../yolo26_pipeline/model/best.pt", conf=0.25, imgsz=512)
# DETECTOR = F.UltralyticsModel("yoloe-26m-seg.pt", text="grape cluster", conf=0.2)   # zero-shot, downloads the weights

# The fine-tuned SAM 2.1 to test: the newest one the team ships in model/ (see model/README.md), else Meta's pretrained
# weights. Set it to another .pt file to test a different checkpoint (training writes them to data/models/).
_shipped = sorted(Path("model").glob("sam2.1_*.pt"), key=lambda p: p.stat().st_mtime)
FINETUNED = _shipped[-1] if _shipped else None

PIPELINE = F.DetectThenSAM2(DETECTOR, ckpt=FINETUNED)
PIPELINE.prepare()              # loads the detector and SAM 2.1 now, so model loading is not timed as inference
print("detector:", Path(DETECTOR.weights).name, "· SAM 2.1:", FINETUNED.name if FINETUNED else f"{F.SAM2_SIZE} pretrained")

In [ ]:
# Step 3: the hand counts (grape clusters per photo), the same list as yolo26_pipeline/count_test.py.
GROUND_TRUTH = [
    ("SYH_2017-04-27_1322.jpg", 20),
    ("SYH_2017-04-27_1318.jpg", 8),
    ("SYH_2017-04-27_1304.jpg", 7),
    ("SYH_2017-04-27_1269.jpg", 11),
    ("SYH_2017-04-27_1239.jpg", 21),
    ("SVB_20180427_151818928.jpg", 21),
    ("SVB_1972.jpg", 22),
    ("SVB_1954.jpg", 19),
    ("SVB_1944.jpg", 18),
    ("SVB_1935.jpg", 25),
    ("CSV_20180427_144723166_HDR.jpg", 14),
    ("CSV_20180427_144535647.jpg", 13),
    ("CSV_20180427_144507419.jpg", 16),
    ("CSV_1898.jpg", 11),
    ("CSV_1877.jpg", 8),
    ("CFR_1666.jpg", 12),
    ("CFR_1651.jpg", 14),
    ("CFR_1643.jpg", 15),
    ("CFR_1641.jpg", 30),
    ("CFR_1639.jpg", 17),
    ("CFR_1638.jpg", 10),
    ("CDY_2054.jpg", 16),
    ("CDY_2051.jpg", 11),
    ("CDY_2046.jpg", 10),
    ("CDY_2043.jpg", 10),
    ("CDY_2040.jpg", 8),
    ("CDY_2015.jpg", 20),
]
print(len(GROUND_TRUTH), "photos,", sum(n for _, n in GROUND_TRUTH), "clusters counted by hand")

## Count every photo
The pipeline reads *clips*: a folder of frames named `00000.jpg`, `00001.jpg`, … (that is also how Meta's SAM 2 reads video). A photo becomes a one-frame clip. For each photo the cell records the detector's box count, the number of objects in SAM 2.1's mask, and saves the mask drawn on the photo to `data/count_test/results/overlays/`.

In [ ]:
# Step 4: run the detector and the detector → SAM 2.1 pipeline on every photo and count the clusters.
STILLS = F.ROOT / "stills"                       # one single-frame clip per photo
OVERLAYS = F.RESULTS / "overlays"
OVERLAYS.mkdir(parents=True, exist_ok=True)

def still_clip(photo):
    """Turn one photo into a one-frame clip the pipeline can read."""
    d = STILLS / photo.stem
    d.mkdir(parents=True, exist_ok=True)
    frame = d / "00000.jpg"
    if not frame.exists():
        shutil.copy2(photo, frame)
    return F.Clip(photo.stem, root=STILLS)

rows = []
for filename, truth in tqdm(GROUND_TRUTH):
    photo = IMAGES / filename
    if not photo.exists():
        rows.append(dict(image=filename, truth=truth, status="missing image"))
        continue
    try:
        clip = still_clip(photo)
        boxes, scores, _ = DETECTOR.detect(clip.image(0))        # the detector alone: one box per cluster it finds
        ids = PIPELINE.predict(clip)                             # detector boxes → SAM 2.1 masks: [1, H, W] object ids, 0 = background
        objects = np.unique(ids[0])
        n_sam2 = int((objects != 0).sum())
        Image.fromarray(F.overlay(clip.image(0), ids[0])).save(OVERLAYS / filename)
        rows.append(dict(image=filename, truth=truth, yolo=len(boxes), sam2=n_sam2, status="OK"))
    except Exception as e:
        rows.append(dict(image=filename, truth=truth, status=f"{type(e).__name__}: {e}"))

df = pd.DataFrame(rows, columns=["image", "truth", "yolo", "sam2", "status"])
df["yolo_error"] = df["yolo"] - df["truth"]
df["sam2_error"] = df["sam2"] - df["truth"]
df.to_csv(F.RESULTS / "count_test.csv", index=False)
display(df)

In [ ]:
# Step 5: count metrics for the detector alone and for detector → SAM 2.1 (same photos, same hand counts).
def count_metrics(pred, truth):
    pred, truth = np.asarray(pred, float), np.asarray(truth, float)
    err = pred - truth
    return {
        "photos": len(pred),
        "MAE": np.abs(err).mean(),
        "RMSE": np.sqrt((err ** 2).mean()),
        "bias (predicted − true)": err.mean(),
        "exact count (%)": (err == 0).mean() * 100,
        "clusters predicted": pred.sum(),
        "clusters by hand": truth.sum(),
    }

ok = df[df["status"] == "OK"]
sam2_name = f"SAM 2.1 {FINETUNED.stem.split('_', 2)[-1] if FINETUNED else 'pretrained'}"
metrics = pd.DataFrame({
    "YOLO26 boxes": count_metrics(ok["yolo"], ok["truth"]),
    f"YOLO26 → {sam2_name}": count_metrics(ok["sam2"], ok["truth"]),
}).round(3)
metrics.to_csv(F.RESULTS / "count_metrics.csv")
display(metrics)
if len(ok) < len(df):
    display(df[df["status"] != "OK"][["image", "status"]])

In [ ]:
# Step 6: look at four photos: SAM 2.1's masks, each cluster numbered, with the three counts in the title.
import matplotlib.pyplot as plt

assert len(ok), "no photo was counted; see the status column above"
picks = ok.iloc[np.linspace(0, len(ok) - 1, min(4, len(ok))).round().astype(int)]
fig, axes = plt.subplots(1, len(picks), figsize=(18, 4.2), squeeze=False)
for ax, r in zip(axes[0], picks.itertuples()):
    ax.imshow(Image.open(OVERLAYS / r.image))
    ax.set_title(f"{r.image}\nby hand {r.truth} · YOLO26 {r.yolo} · SAM 2.1 {r.sam2}", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

## Reading the numbers
- **On single photos, SAM 2.1 cannot add clusters**: every object it draws started as a detector box. Its count is at most the detector's. It is lower when two boxes overlap (the pipeline merges boxes whose overlap is at least `match_iou=0.4`, or where 70% of the smaller box lies inside the larger) or when SAM 2.1 returns an empty mask for a box.
- **Where SAM 2.1 earns its place is video**: it follows each cluster from frame to frame, so a cluster that the detector finds on 30 frames is counted once. That is the comparison that matters for the yield estimate, and `python fruit.py evaluate` makes it on the labelled held-out clips (the *automatic* leaderboard: F1 and counts). This notebook is the still-photo check that the plumbing works and the masks look right.
- To test a fine-tuned SAM 2.1, put its `.pt` in `model/` (Step 2 picks the newest one up by itself), or set `FINETUNED` in Step 2 to any checkpoint file (training writes them to `data/models/`).